# 천안시 SGIS 공식 격자별 최종 데이터 생성

100m, 500m, 1km를 하나의 코드로 순서대로 실행합니다.

입력은 `/content/drive/MyDrive/Cheonan/grid_inputs`, 결과는 `/content/drive/MyDrive/Cheonan/grid_outputs`에 저장됩니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!pip -q install openpyxl


## 입력 폴더 구조

```text
Cheonan/
  grid_inputs/
    sgis_population/
      2024년_인구_다바_100M.csv
      2024년_인구_다바_500M.csv
      2024년_인구_다바_1K.csv
    sgis_grid_border/
      grid_다바_100M.*
      grid_다바_500M.*
      grid_다바_1K.*
    cheonan_adm_dong_boundary_20250630.zip
    population-may.xlsx
    commerce_cheonan_202603.csv
    hospital_cheonan_202603.csv
    pharmacy_cheonan_202603.csv
    cheonan_address_facility_rows.csv
    cheonan_address_geocoding_cache.csv
```


In [ ]:
from __future__ import annotations

import io
import json
import math
import re
import struct
import unicodedata
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd


WORKSPACE = Path("/content/drive/MyDrive/Cheonan")
GRID_INPUT_DIR = WORKSPACE / "grid_inputs"
OUTPUT_DIR = WORKSPACE / "grid_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SGIS_POPULATION_DIR = GRID_INPUT_DIR / "sgis_population"
SGIS_GRID_BORDER_DIR = GRID_INPUT_DIR / "sgis_grid_border"

BOUNDARY_ZIP = GRID_INPUT_DIR / "cheonan_adm_dong_boundary_20250630.zip"
POPULATION_XLSX = GRID_INPUT_DIR / "population-may.xlsx"

HYEONSEOK_DIR = GRID_INPUT_DIR
ADDRESS_FACILITY_ROWS = GRID_INPUT_DIR / "cheonan_address_facility_rows.csv"
ADDRESS_GEOCODE_CACHE = GRID_INPUT_DIR / "cheonan_address_geocoding_cache.csv"

GRID_RUNS = [
    {"label": "100m", "grid_m": 100, "sgis_size": "100M", "output_tag": "100m"},
    {"label": "500m", "grid_m": 500, "sgis_size": "500M", "output_tag": "500m"},
    {"label": "1km", "grid_m": 1000, "sgis_size": "1K", "output_tag": "1km"},
]

GRID_M = 100
SGIS_SIZE = "100M"
PREFIX = OUTPUT_DIR / "cheonan_sgis_100m"


def safe_name(value: object, max_len: int = 48) -> str:
    text = unicodedata.normalize("NFC", str(value).strip())
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"[^\w]+", "_", text, flags=re.UNICODE)
    text = re.sub(r"_+", "_", text).strip("_")
    return (text or "unknown")[:max_len]


def normalize_adm_name(value: object) -> str:
    return re.sub(r"\s+", "", str(value).strip())


def read_csv_any(path: Path, nrows: int | None = None) -> tuple[pd.DataFrame, str]:
    last_error: Exception | None = None
    for enc in ("utf-8-sig", "utf-8", "cp949", "euc-kr"):
        try:
            return pd.read_csv(path, encoding=enc, nrows=nrows), enc
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Could not read CSV {path}: {last_error}")


def dbf_records(dbf_bytes: bytes, encoding: str) -> list[dict[str, str]]:
    nrec = struct.unpack("<I", dbf_bytes[4:8])[0]
    header_len = struct.unpack("<H", dbf_bytes[8:10])[0]
    rec_len = struct.unpack("<H", dbf_bytes[10:12])[0]
    fields: list[tuple[str, int]] = []
    off = 32
    while dbf_bytes[off] != 0x0D:
        desc = dbf_bytes[off : off + 32]
        name = desc[:11].split(b"\x00", 1)[0].decode("ascii", "ignore")
        size = desc[16]
        fields.append((name, size))
        off += 32
    rows = []
    for i in range(nrec):
        row = dbf_bytes[header_len + i * rec_len : header_len + (i + 1) * rec_len]
        pos = 1
        values = {}
        for name, size in fields:
            raw = row[pos : pos + size]
            pos += size
            values[name] = raw.decode(encoding, "ignore").strip()
        rows.append(values)
    return rows


def read_boundary_shapes(boundary_zip: Path = BOUNDARY_ZIP) -> list[dict[str, object]]:
    with zipfile.ZipFile(boundary_zip) as z:
        shp_name = next(n for n in z.namelist() if n.endswith(".shp"))
        dbf_name = next(n for n in z.namelist() if n.endswith(".dbf"))
        shp_bytes = z.read(shp_name)
        attrs = dbf_records(z.read(dbf_name), "cp949")

    shapes: list[dict[str, object]] = []
    off = 100
    while off + 8 <= len(shp_bytes):
        rec_no, content_words = struct.unpack(">2i", shp_bytes[off : off + 8])
        off += 8
        content = memoryview(shp_bytes)[off : off + content_words * 2]
        off += content_words * 2
        if len(content) < 44 or struct.unpack("<i", content[0:4])[0] != 5:
            continue
        bbox = struct.unpack("<4d", content[4:36])
        num_parts, num_points = struct.unpack("<2i", content[36:44])
        part_start = 44
        parts = list(struct.unpack("<" + "i" * num_parts, content[part_start : part_start + 4 * num_parts]))
        point_start = part_start + 4 * num_parts
        points = [
            struct.unpack("<2d", content[point_start + 16 * i : point_start + 16 * (i + 1)])
            for i in range(num_points)
        ]
        rings = []
        for i, start in enumerate(parts):
            end = parts[i + 1] if i + 1 < len(parts) else num_points
            ring = points[start:end]
            if len(ring) < 3:
                continue
            xs = [p[0] for p in ring]
            ys = [p[1] for p in ring]
            rings.append({"points": ring, "bbox": (min(xs), min(ys), max(xs), max(ys))})
        shapes.append({"attrs": attrs[rec_no - 1], "bbox": bbox, "rings": rings})
    return shapes


def point_in_ring(x: float, y: float, ring: list[tuple[float, float]]) -> bool:
    inside = False
    j = len(ring) - 1
    for i, (xi, yi) in enumerate(ring):
        xj, yj = ring[j]
        if (yi > y) != (yj > y):
            x_inter = (xj - xi) * (y - yi) / ((yj - yi) or 1e-30) + xi
            if x < x_inter:
                inside = not inside
        j = i
    return inside


def point_in_shape(x: float, y: float, shape: dict[str, object]) -> bool:
    minx, miny, maxx, maxy = shape["bbox"]  # type: ignore[misc]
    if x < minx or x > maxx or y < miny or y > maxy:
        return False
    inside = False
    for ring in shape["rings"]:  # type: ignore[index]
        rx0, ry0, rx1, ry1 = ring["bbox"]
        if x < rx0 or x > rx1 or y < ry0 or y > ry1:
            continue
        if point_in_ring(x, y, ring["points"]):
            inside = not inside
    return inside


def find_admin_for_point(x: float, y: float, shapes: list[dict[str, object]]) -> dict[str, str] | None:
    for shape in shapes:
        if point_in_shape(x, y, shape):
            return shape["attrs"]  # type: ignore[return-value]
    return None


def meridional_arc(phi: np.ndarray, a: float, e2: float) -> np.ndarray:
    e4 = e2 * e2
    e6 = e4 * e2
    return a * (
        (1 - e2 / 4 - 3 * e4 / 64 - 5 * e6 / 256) * phi
        - (3 * e2 / 8 + 3 * e4 / 32 + 45 * e6 / 1024) * np.sin(2 * phi)
        + (15 * e4 / 256 + 45 * e6 / 1024) * np.sin(4 * phi)
        - (35 * e6 / 3072) * np.sin(6 * phi)
    )


def tm_forward(
    lon: np.ndarray,
    lat: np.ndarray,
    *,
    lon0_deg: float,
    lat0_deg: float,
    false_easting: float,
    false_northing: float,
    scale_factor: float,
) -> tuple[np.ndarray, np.ndarray]:
    a = 6378137.0
    inv_f = 298.257222101
    f = 1.0 / inv_f
    e2 = f * (2.0 - f)
    ep2 = e2 / (1.0 - e2)

    phi = np.radians(lat.astype(float))
    lam = np.radians(lon.astype(float))
    lon0 = math.radians(lon0_deg)
    lat0 = math.radians(lat0_deg)

    sin_phi = np.sin(phi)
    cos_phi = np.cos(phi)
    tan_phi = np.tan(phi)
    n = a / np.sqrt(1 - e2 * sin_phi * sin_phi)
    t = tan_phi * tan_phi
    c = ep2 * cos_phi * cos_phi
    aa = (lam - lon0) * cos_phi
    m = meridional_arc(phi, a, e2)
    m0 = float(meridional_arc(np.array([lat0]), a, e2)[0])

    x = false_easting + scale_factor * n * (
        aa
        + (1 - t + c) * aa**3 / 6
        + (5 - 18 * t + t**2 + 72 * c - 58 * ep2) * aa**5 / 120
    )
    y = false_northing + scale_factor * (
        m
        - m0
        + n
        * tan_phi
        * (
            aa**2 / 2
            + (5 - t + 9 * c + 4 * c**2) * aa**4 / 24
            + (61 - 58 * t + t**2 + 600 * c - 330 * ep2) * aa**6 / 720
        )
    )
    return x, y


def tm_inverse(
    x: np.ndarray,
    y: np.ndarray,
    *,
    lon0_deg: float,
    lat0_deg: float,
    false_easting: float,
    false_northing: float,
    scale_factor: float,
) -> tuple[np.ndarray, np.ndarray]:
    a = 6378137.0
    inv_f = 298.257222101
    f = 1.0 / inv_f
    e2 = f * (2.0 - f)
    ep2 = e2 / (1.0 - e2)
    e1 = (1 - math.sqrt(1 - e2)) / (1 + math.sqrt(1 - e2))

    lat0 = math.radians(lat0_deg)
    lon0 = math.radians(lon0_deg)
    m0 = float(meridional_arc(np.array([lat0]), a, e2)[0])
    m = m0 + (y.astype(float) - false_northing) / scale_factor
    mu = m / (a * (1 - e2 / 4 - 3 * e2**2 / 64 - 5 * e2**3 / 256))

    phi1 = (
        mu
        + (3 * e1 / 2 - 27 * e1**3 / 32) * np.sin(2 * mu)
        + (21 * e1**2 / 16 - 55 * e1**4 / 32) * np.sin(4 * mu)
        + (151 * e1**3 / 96) * np.sin(6 * mu)
        + (1097 * e1**4 / 512) * np.sin(8 * mu)
    )
    sin1 = np.sin(phi1)
    cos1 = np.cos(phi1)
    tan1 = np.tan(phi1)
    n1 = a / np.sqrt(1 - e2 * sin1 * sin1)
    r1 = a * (1 - e2) / (1 - e2 * sin1 * sin1) ** 1.5
    t1 = tan1 * tan1
    c1 = ep2 * cos1 * cos1
    d = (x.astype(float) - false_easting) / (n1 * scale_factor)

    lat = phi1 - (n1 * tan1 / r1) * (
        d**2 / 2
        - (5 + 3 * t1 + 10 * c1 - 4 * c1**2 - 9 * ep2) * d**4 / 24
        + (61 + 90 * t1 + 298 * c1 + 45 * t1**2 - 252 * ep2 - 3 * c1**2) * d**6 / 720
    )
    lon = lon0 + (
        d
        - (1 + 2 * t1 + c1) * d**3 / 6
        + (5 - 2 * c1 + 28 * t1 - 3 * c1**2 + 8 * ep2 + 24 * t1**2) * d**5 / 120
    ) / cos1
    return np.degrees(lon), np.degrees(lat)


def lonlat_to_central(lon: np.ndarray, lat: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    return tm_forward(
        lon,
        lat,
        lon0_deg=127.0,
        lat0_deg=38.0,
        false_easting=200000.0,
        false_northing=600000.0,
        scale_factor=1.0,
    )


def central_to_lonlat(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    return tm_inverse(
        x,
        y,
        lon0_deg=127.0,
        lat0_deg=38.0,
        false_easting=200000.0,
        false_northing=600000.0,
        scale_factor=1.0,
    )


def lonlat_to_unified(lon: np.ndarray, lat: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    return tm_forward(
        lon,
        lat,
        lon0_deg=127.5,
        lat0_deg=38.0,
        false_easting=1000000.0,
        false_northing=2000000.0,
        scale_factor=0.9996,
    )


def unified_to_lonlat(x: np.ndarray, y: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    return tm_inverse(
        x,
        y,
        lon0_deg=127.5,
        lat0_deg=38.0,
        false_easting=1000000.0,
        false_northing=2000000.0,
        scale_factor=0.9996,
    )


def read_sgis_grid_bboxes(grid_dir: Path | None = None, sgis_size: str | None = None) -> pd.DataFrame:
    grid_dir = grid_dir or SGIS_GRID_BORDER_DIR
    sgis_size = sgis_size or SGIS_SIZE
    shp_path = next(grid_dir.glob(f"grid_*_{sgis_size}.shp"))
    dbf_path = next(grid_dir.glob(f"grid_*_{sgis_size}.dbf"))
    shp_bytes = shp_path.read_bytes()
    attrs = dbf_records(dbf_path.read_bytes(), "utf-8")

    rows = []
    off = 100
    while off + 8 <= len(shp_bytes):
        rec_no, content_words = struct.unpack(">2i", shp_bytes[off : off + 8])
        off += 8
        content = memoryview(shp_bytes)[off : off + content_words * 2]
        off += content_words * 2
        if len(content) < 36 or struct.unpack("<i", content[0:4])[0] != 5:
            continue
        minx, miny, maxx, maxy = struct.unpack("<4d", content[4:36])
        rows.append(
            {
                "GRID_CD": attrs[rec_no - 1]["GRID_CD"],
                "x_min": float(minx),
                "y_min": float(miny),
                "x_max": float(maxx),
                "y_max": float(maxy),
                "center_x": float((minx + maxx) / 2),
                "center_y": float((miny + maxy) / 2),
                "grid_m": GRID_M,
                "selection": "official_sgis_100m_center_in_cheonan",
            }
        )
    return pd.DataFrame(rows)


def cheonan_bbox(shapes: list[dict[str, object]]) -> tuple[float, float, float, float]:
    minx = min(shape["bbox"][0] for shape in shapes)  # type: ignore[index]
    miny = min(shape["bbox"][1] for shape in shapes)  # type: ignore[index]
    maxx = max(shape["bbox"][2] for shape in shapes)  # type: ignore[index]
    maxy = max(shape["bbox"][3] for shape in shapes)  # type: ignore[index]
    return float(minx), float(miny), float(maxx), float(maxy)


def build_cheonan_official_grid(shapes: list[dict[str, object]]) -> pd.DataFrame:
    grid = read_sgis_grid_bboxes()
    lon, lat = unified_to_lonlat(grid["center_x"].to_numpy(), grid["center_y"].to_numpy())
    cx_central, cy_central = lonlat_to_central(lon, lat)
    grid["center_lon"] = lon
    grid["center_lat"] = lat
    grid["center_x_central"] = cx_central
    grid["center_y_central"] = cy_central

    bx0, by0, bx1, by1 = cheonan_bbox(shapes)
    candidate = grid[
        grid["center_x_central"].between(bx0 - 1000, bx1 + 1000)
        & grid["center_y_central"].between(by0 - 1000, by1 + 1000)
    ].copy()

    adm_cd = []
    adm_nm = []
    for x, y in zip(candidate["center_x_central"].to_numpy(), candidate["center_y_central"].to_numpy()):
        admin = find_admin_for_point(float(x), float(y), shapes)
        adm_cd.append(admin.get("ADM_CD", "") if admin else "")
        adm_nm.append(admin.get("ADM_NM", "") if admin else "")
    candidate["center_adm_cd"] = adm_cd
    candidate["center_adm_nm"] = adm_nm
    candidate = candidate[candidate["center_adm_cd"].astype(str).ne("")].copy()
    candidate["adm_nm_norm"] = candidate["center_adm_nm"].map(normalize_adm_name)
    return candidate.reset_index(drop=True)


def read_sgis_2024_population(pop_path: Path | None = None, sgis_size: str | None = None) -> pd.DataFrame:
    sgis_size = sgis_size or SGIS_SIZE
    pop_path = pop_path or next(SGIS_POPULATION_DIR.glob(f"*_{sgis_size}.csv"))
    df = pd.read_csv(
        pop_path,
        encoding="cp949",
        header=None,
        names=["year", "GRID_CD", "stat_code", "value"],
    )
    pop = df[df["stat_code"].eq("to_in_001")].copy()
    pop["pop_2024_grid"] = pd.to_numeric(pop["value"], errors="coerce").fillna(0)
    return pop[["GRID_CD", "pop_2024_grid"]]


def read_2026_adm_population(path: Path = POPULATION_XLSX) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name="행정통(리)인구", header=None, skiprows=4)
    rows = df[(df[1].astype(str).str.strip() == "계") & df[0].notna()].copy()
    rows = rows[[0, 2, 3, 4, 5]].copy()
    rows.columns = ["center_adm_nm", "households_2026_adm", "pop_2026_adm", "pop_male_2026_adm", "pop_female_2026_adm"]
    for col in ["households_2026_adm", "pop_2026_adm", "pop_male_2026_adm", "pop_female_2026_adm"]:
        rows[col] = pd.to_numeric(rows[col], errors="coerce").fillna(0).astype(int)
    rows["adm_nm_norm"] = rows["center_adm_nm"].map(normalize_adm_name)
    return rows


def shape_representative_point(shape: dict[str, object]) -> tuple[float, float]:
    minx, miny, maxx, maxy = shape["bbox"]  # type: ignore[misc]
    candidates: list[tuple[float, float]] = [((minx + maxx) / 2, (miny + maxy) / 2)]
    for ring in shape["rings"]:  # type: ignore[index]
        pts = ring["points"]
        if pts:
            xs = [p[0] for p in pts]
            ys = [p[1] for p in pts]
            candidates.append((float(sum(xs) / len(xs)), float(sum(ys) / len(ys))))
            candidates.append(pts[0])
    for x, y in candidates:
        if point_in_shape(float(x), float(y), shape):
            return float(x), float(y)
    return float((minx + maxx) / 2), float((miny + maxy) / 2)


def nearest_grid_index_for_central_point(grid: pd.DataFrame, x_central: float, y_central: float) -> int:
    lon, lat = central_to_lonlat(np.array([x_central]), np.array([y_central]))
    ux, uy = lonlat_to_unified(lon, lat)
    key_x = int(math.floor(float(ux[0]) / GRID_M) * GRID_M)
    key_y = int(math.floor(float(uy[0]) / GRID_M) * GRID_M)
    lookup = official_grid_lookup(grid)
    grid_cd = lookup.get((key_x, key_y), "")
    if grid_cd:
        match = grid.index[grid["GRID_CD"].eq(grid_cd)]
        if len(match):
            return int(match[0])
    dx = grid["center_x"].to_numpy(float) - float(ux[0])
    dy = grid["center_y"].to_numpy(float) - float(uy[0])
    return int(np.argmin(dx * dx + dy * dy))


def allocate_largest_remainder(df: pd.DataFrame, group_col: str, float_col: str, target_col: str, out_col: str) -> pd.Series:
    out = pd.Series(0, index=df.index, dtype="int64")
    for _, idx in df.groupby(group_col, sort=False).groups.items():
        sub = df.loc[idx]
        target = int(sub[target_col].iloc[0]) if len(sub) else 0
        values = pd.to_numeric(sub[float_col], errors="coerce").fillna(0).to_numpy(float)
        floors = np.floor(values).astype(int)
        diff = target - int(floors.sum())
        ints = floors.copy()
        if diff > 0 and len(ints) > 0:
            order = np.argsort(-(values - floors), kind="mergesort")
            ints[order[:diff]] += 1
        elif diff < 0 and len(ints) > 0:
            order = np.argsort(values - floors, kind="mergesort")
            take = min(-diff, len(ints))
            ints[order[:take]] -= 1
        out.loc[idx] = ints
    return out.rename(out_col)


def add_population_estimates(
    grid: pd.DataFrame,
    shapes: list[dict[str, object]] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    sgis_pop = read_sgis_2024_population()
    adm_pop = read_2026_adm_population()
    merged = grid.merge(sgis_pop, on="GRID_CD", how="left")
    merged["pop_2024_grid"] = pd.to_numeric(merged["pop_2024_grid"], errors="coerce").fillna(0)
    merged = merged.merge(
        adm_pop.drop(columns=["center_adm_nm"]),
        on="adm_nm_norm",
        how="left",
        validate="many_to_one",
    )
    for col in ["households_2026_adm", "pop_2026_adm", "pop_male_2026_adm", "pop_female_2026_adm"]:
        merged[col] = pd.to_numeric(merged[col], errors="coerce").fillna(0).astype(int)

    merged["pop_2024_adm_sum"] = merged.groupby("adm_nm_norm")["pop_2024_grid"].transform("sum")
    merged["grid_count_in_adm"] = merged.groupby("adm_nm_norm")["GRID_CD"].transform("count")
    positive = merged["pop_2024_adm_sum"].gt(0)
    merged["pop_ratio_in_adm_2024"] = np.where(
        positive,
        merged["pop_2024_grid"] / merged["pop_2024_adm_sum"],
        1.0 / merged["grid_count_in_adm"].replace(0, np.nan),
    )
    merged["pop_ratio_in_adm_2024"] = merged["pop_ratio_in_adm_2024"].fillna(0)

    for target_col, prefix in [
        ("pop_2026_adm", "pop_2026_est"),
        ("pop_male_2026_adm", "pop_male_2026_est"),
        ("pop_female_2026_adm", "pop_female_2026_est"),
        ("households_2026_adm", "households_2026_est"),
    ]:
        merged[prefix] = merged["pop_ratio_in_adm_2024"] * merged[target_col]
        merged[f"{prefix}_int"] = allocate_largest_remainder(merged, "adm_nm_norm", prefix, target_col, f"{prefix}_int")

    merged["fallback_admin_population_added"] = 0
    merged["fallback_admin_names"] = ""
    for prefix in ["pop_2026_est", "pop_male_2026_est", "pop_female_2026_est", "households_2026_est"]:
        merged[f"fallback_{prefix}_added"] = 0
    fallback_summary_rows: list[dict[str, object]] = []
    missing_adm = adm_pop[~adm_pop["adm_nm_norm"].isin(set(merged["adm_nm_norm"]))]
    if not missing_adm.empty:
        if shapes is None:
            shapes = read_boundary_shapes()
        shapes_by_norm = {normalize_adm_name(shape["attrs"].get("ADM_NM", "")): shape for shape in shapes}  # type: ignore[index]
        for _, row in missing_adm.iterrows():
            adm_norm = row["adm_nm_norm"]
            shape = shapes_by_norm.get(adm_norm)
            if shape is not None:
                x_rep, y_rep = shape_representative_point(shape)
                target_idx = nearest_grid_index_for_central_point(merged, x_rep, y_rep)
                adm_cd = shape["attrs"].get("ADM_CD", "")  # type: ignore[index]
                adm_nm = shape["attrs"].get("ADM_NM", row["center_adm_nm"])  # type: ignore[index]
            else:
                target_idx = int(merged.index[0])
                adm_cd = ""
                adm_nm = row["center_adm_nm"]
            for target_col, prefix in [
                ("pop_2026_adm", "pop_2026_est"),
                ("pop_male_2026_adm", "pop_male_2026_est"),
                ("pop_female_2026_adm", "pop_female_2026_est"),
                ("households_2026_adm", "households_2026_est"),
            ]:
                value = int(row[target_col])
                merged.loc[target_idx, prefix] += value
                merged.loc[target_idx, f"{prefix}_int"] += value
                merged.loc[target_idx, f"fallback_{prefix}_added"] += value
            merged.loc[target_idx, "fallback_admin_population_added"] += int(row["pop_2026_adm"])
            current_names = str(merged.loc[target_idx, "fallback_admin_names"] or "")
            merged.loc[target_idx, "fallback_admin_names"] = (current_names + ";" if current_names else "") + str(row["center_adm_nm"])
            fallback_summary_rows.append(
                {
                    "center_adm_cd": adm_cd,
                    "center_adm_nm": adm_nm,
                    "grid_count": 0,
                    "pop_2024_grid_sum": 0.0,
                    "pop_ratio_sum": 0.0,
                    "pop_2026_adm": int(row["pop_2026_adm"]),
                    "pop_2026_est_sum": float(row["pop_2026_adm"]),
                    "pop_2026_est_int_sum": int(row["pop_2026_adm"]),
                    "households_2026_adm": int(row["households_2026_adm"]),
                    "households_2026_est_int_sum": int(row["households_2026_adm"]),
                    "allocation_method": "fallback_admin_representative_point",
                    "fallback_grid_cd": merged.loc[target_idx, "GRID_CD"],
                }
            )

    merged["_pop_2026_est_summary"] = merged["pop_2026_est"] - merged["fallback_pop_2026_est_added"]
    merged["_pop_2026_est_int_summary"] = merged["pop_2026_est_int"] - merged["fallback_pop_2026_est_added"]
    merged["_households_2026_est_int_summary"] = (
        merged["households_2026_est_int"] - merged["fallback_households_2026_est_added"]
    )

    adm_summary = (
        merged.groupby(["center_adm_cd", "center_adm_nm"], dropna=False)
        .agg(
            grid_count=("GRID_CD", "count"),
            pop_2024_grid_sum=("pop_2024_grid", "sum"),
            pop_ratio_sum=("pop_ratio_in_adm_2024", "sum"),
            pop_2026_adm=("pop_2026_adm", "first"),
            pop_2026_est_sum=("_pop_2026_est_summary", "sum"),
            pop_2026_est_int_sum=("_pop_2026_est_int_summary", "sum"),
            households_2026_adm=("households_2026_adm", "first"),
            households_2026_est_int_sum=("_households_2026_est_int_summary", "sum"),
        )
        .reset_index()
    )
    merged = merged.drop(
        columns=["_pop_2026_est_summary", "_pop_2026_est_int_summary", "_households_2026_est_int_summary"]
    )
    adm_summary["allocation_method"] = "center_in_ratio"
    adm_summary["fallback_grid_cd"] = ""
    if fallback_summary_rows:
        adm_summary = pd.concat([adm_summary, pd.DataFrame(fallback_summary_rows)], ignore_index=True, sort=False)
    return merged, adm_summary


def official_grid_lookup(grid: pd.DataFrame) -> dict[tuple[int, int], str]:
    return {
        (int(round(row.x_min)), int(round(row.y_min))): row.GRID_CD
        for row in grid[["GRID_CD", "x_min", "y_min"]].itertuples(index=False)
    }


def assign_lonlat_to_grid(df: pd.DataFrame, lon_col: str, lat_col: str, grid_lookup: dict[tuple[int, int], str]) -> pd.DataFrame:
    out = df.copy()
    lon = pd.to_numeric(out[lon_col], errors="coerce")
    lat = pd.to_numeric(out[lat_col], errors="coerce")
    valid = lon.notna() & lat.notna()
    out["GRID_CD"] = ""
    out["x"] = np.nan
    out["y"] = np.nan
    if valid.any():
        x, y = lonlat_to_unified(lon[valid].to_numpy(float), lat[valid].to_numpy(float))
        keys = [(int(math.floor(xx / GRID_M) * GRID_M), int(math.floor(yy / GRID_M) * GRID_M)) for xx, yy in zip(x, y)]
        out.loc[valid, "x"] = x
        out.loc[valid, "y"] = y
        out.loc[valid, "GRID_CD"] = [grid_lookup.get(key, "") for key in keys]
    return out


def aggregate_coordinate_sources(grid: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    lookup = official_grid_lookup(grid)
    feature_tables = []
    assignment_tables = []
    summary_rows = []
    specs = [
        {
            "source_id": "commerce",
            "path": HYEONSEOK_DIR / "commerce_cheonan_202603.csv",
            "lon_col": "경도",
            "lat_col": "위도",
            "name_col": "상호명",
        },
        {
            "source_id": "hospital",
            "path": HYEONSEOK_DIR / "hospital_cheonan_202603.csv",
            "lon_col": "좌표(X)",
            "lat_col": "좌표(Y)",
            "name_col": "요양기관명",
        },
        {
            "source_id": "pharmacy",
            "path": HYEONSEOK_DIR / "pharmacy_cheonan_202603.csv",
            "lon_col": "좌표(X)",
            "lat_col": "좌표(Y)",
            "name_col": "약국명",
        },
    ]
    for spec in specs:
        path = spec["path"]
        if not path.exists():
            summary_rows.append({"source_id": spec["source_id"], "file_name": path.name, "status": "missing"})
            continue
        df, enc = read_csv_any(path)
        df = df.reset_index().rename(columns={"index": "source_row"})
        df["source_row"] = df["source_row"] + 2
        assigned = assign_lonlat_to_grid(df, spec["lon_col"], spec["lat_col"], lookup)
        assigned["source_id"] = spec["source_id"]
        assigned["source_file"] = path.name
        assigned["name"] = assigned.get(spec["name_col"], pd.Series([""] * len(assigned))).fillna("")
        cols = ["source_id", "source_file", "source_row", "GRID_CD", "name", spec["lon_col"], spec["lat_col"], "x", "y"]
        assignment_tables.append(assigned[[c for c in cols if c in assigned.columns]].rename(columns={spec["lon_col"]: "lon", spec["lat_col"]: "lat"}))

        ok = assigned[assigned["GRID_CD"].astype(str).ne("")].copy()
        features = ok.groupby("GRID_CD").size().rename(f"{spec['source_id']}_total").reset_index()
        if spec["source_id"] == "commerce":
            for col, prefix in [("상권업종대분류명", "commerce_major"), ("상권업종중분류명", "commerce_middle")]:
                if col in ok.columns:
                    pivot = (
                        ok.pivot_table(index="GRID_CD", columns=col, values="source_row", aggfunc="count", fill_value=0)
                        .rename(columns=lambda v: f"{prefix}_{safe_name(v)}")
                        .reset_index()
                    )
                    features = features.merge(pivot, on="GRID_CD", how="left")
        elif spec["source_id"] == "hospital":
            if "종별코드명" in ok.columns:
                pivot = (
                    ok.pivot_table(index="GRID_CD", columns="종별코드명", values="source_row", aggfunc="count", fill_value=0)
                    .rename(columns=lambda v: f"hospital_type_{safe_name(v)}")
                    .reset_index()
                )
                features = features.merge(pivot, on="GRID_CD", how="left")
            if "총의사수" in ok.columns:
                doctor = (
                    ok.assign(_num=pd.to_numeric(ok["총의사수"], errors="coerce").fillna(0))
                    .groupby("GRID_CD")["_num"]
                    .sum()
                    .rename("hospital_doctor_sum")
                    .reset_index()
                )
                features = features.merge(doctor, on="GRID_CD", how="left")
        feature_tables.append(features)
        summary_rows.append(
            {
                "source_id": spec["source_id"],
                "file_name": path.name,
                "encoding": enc,
                "status": "processed",
                "rows": int(len(df)),
                "valid_coordinate_rows": int(pd.to_numeric(df[spec["lon_col"]], errors="coerce").notna().sum()),
                "matched_grid_rows": int(len(ok)),
                "unmatched_grid_rows": int(len(assigned) - len(ok)),
            }
        )
    base = grid[["GRID_CD"]].copy()
    for features in feature_tables:
        base = base.merge(features, on="GRID_CD", how="left")
    for col in base.columns:
        if col != "GRID_CD":
            base[col] = pd.to_numeric(base[col], errors="coerce").fillna(0).astype(int)
    assignments = pd.concat(assignment_tables, ignore_index=True, sort=False) if assignment_tables else pd.DataFrame()
    return base, assignments, pd.DataFrame(summary_rows)


def aggregate_address_sources(grid: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not ADDRESS_FACILITY_ROWS.exists() or not ADDRESS_GEOCODE_CACHE.exists():
        return grid[["GRID_CD"]].copy(), pd.DataFrame(), pd.DataFrame([{"source_id": "address_sources", "status": "missing"}])
    lookup = official_grid_lookup(grid)
    facility = pd.read_csv(ADDRESS_FACILITY_ROWS, encoding="utf-8-sig")
    cache = pd.read_csv(ADDRESS_GEOCODE_CACHE, encoding="utf-8-sig")
    ok_cache = cache[cache["status"].eq("ok")].sort_values("provider").drop_duplicates("address_key", keep="last")
    joined = facility.merge(ok_cache, on="address_key", how="left")
    joined["lon"] = pd.to_numeric(joined["lon"], errors="coerce")
    joined["lat"] = pd.to_numeric(joined["lat"], errors="coerce")
    assigned = assign_lonlat_to_grid(joined, "lon", "lat", lookup)
    matched = assigned[assigned["GRID_CD"].astype(str).ne("")].copy()

    features = grid[["GRID_CD"]].copy()
    total = matched.groupby("GRID_CD").size().rename("address_facility_total").reset_index()
    features = features.merge(total, on="GRID_CD", how="left")
    if not matched.empty:
        by_source = (
            matched.pivot_table(index="GRID_CD", columns="source_id", values="address_key", aggfunc="count", fill_value=0)
            .rename(columns=lambda v: f"{safe_name(v)}_count")
            .reset_index()
        )
        features = features.merge(by_source, on="GRID_CD", how="left")
        for source_id, sub in matched.groupby("source_id"):
            if "category" in sub.columns and sub["category"].notna().any():
                pivot = (
                    sub.pivot_table(index="GRID_CD", columns="category", values="address_key", aggfunc="count", fill_value=0)
                    .rename(columns=lambda v: f"{safe_name(source_id)}_{safe_name(v)}")
                    .reset_index()
                )
                features = features.merge(pivot, on="GRID_CD", how="left")
    for col in features.columns:
        if col != "GRID_CD":
            features[col] = pd.to_numeric(features[col], errors="coerce").fillna(0).astype(int)
    summary = pd.DataFrame(
        [
            {
                "source_id": "address_sources",
                "status": "processed",
                "facility_rows": int(len(facility)),
                "unique_addresses": int(facility["address_key"].nunique()),
                "geocoded_ok_addresses": int(cache["status"].eq("ok").sum()),
                "geocoded_facility_rows": int(joined["lon"].notna().sum()),
                "matched_grid_rows": int(len(matched)),
                "unmatched_grid_rows": int(len(assigned) - len(matched)),
            }
        ]
    )
    return features, assigned, summary


def write_outputs(
    grid_pop: pd.DataFrame,
    adm_summary: pd.DataFrame,
    coord_features: pd.DataFrame,
    coord_assignments: pd.DataFrame,
    coord_summary: pd.DataFrame,
    address_features: pd.DataFrame,
    address_assignments: pd.DataFrame,
    address_summary: pd.DataFrame,
) -> dict[str, object]:
    final = (
        grid_pop.merge(coord_features, on="GRID_CD", how="left")
        .merge(address_features, on="GRID_CD", how="left")
        .copy()
    )
    numeric_fill = final.select_dtypes(include=["number"]).columns
    final[numeric_fill] = final[numeric_fill].fillna(0)

    source_summary = pd.concat([coord_summary, address_summary], ignore_index=True, sort=False)

    outputs = {
        "grid_csv": Path(f"{PREFIX}_grid_cells_center_in.csv"),
        "population_adm_summary_csv": Path(f"{PREFIX}_population_by_adm_summary.csv"),
        "coordinate_assignments_csv": Path(f"{PREFIX}_coordinate_point_assignments.csv"),
        "address_assignments_csv": Path(f"{PREFIX}_address_point_assignments.csv"),
        "source_summary_csv": Path(f"{PREFIX}_source_summary.csv"),
        "final_csv": Path(f"{PREFIX}_final_features_with_population.csv"),
        "metadata_json": Path(f"{PREFIX}_metadata.json"),
    }
    grid_pop.to_csv(outputs["grid_csv"], index=False, encoding="utf-8-sig")
    adm_summary.to_csv(outputs["population_adm_summary_csv"], index=False, encoding="utf-8-sig")
    coord_assignments.to_csv(outputs["coordinate_assignments_csv"], index=False, encoding="utf-8-sig")
    address_assignments.to_csv(outputs["address_assignments_csv"], index=False, encoding="utf-8-sig")
    source_summary.to_csv(outputs["source_summary_csv"], index=False, encoding="utf-8-sig")
    final.to_csv(outputs["final_csv"], index=False, encoding="utf-8-sig")

    metadata = {
        "grid_basis": f"SGIS official {GRID_M}m grid ({SGIS_SIZE})",
        "population_method": f"2026-05-31 administrative-dong totals allocated by 2024 SGIS {GRID_M}m grid population ratios within each administrative dong",
        "grid_rows": int(len(grid_pop)),
        "final_shape": [int(final.shape[0]), int(final.shape[1])],
        "population_2026_total_int": int(final["pop_2026_est_int"].sum()),
        "population_2026_total_float": float(final["pop_2026_est"].sum()),
        "population_2024_grid_sum": float(final["pop_2024_grid"].sum()),
        "source_summary": source_summary.to_dict(orient="records"),
        "outputs": {key: str(value) for key, value in outputs.items()},
    }
    outputs["metadata_json"].write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    return metadata

def run_one_grid(config: dict[str, object], shapes: list[dict[str, object]]) -> dict[str, object]:
    global GRID_M, SGIS_SIZE, PREFIX
    GRID_M = int(config["grid_m"])
    SGIS_SIZE = str(config["sgis_size"])
    PREFIX = OUTPUT_DIR / f"cheonan_sgis_{config['output_tag']}"

    grid = build_cheonan_official_grid(shapes)
    grid_pop, adm_summary = add_population_estimates(grid)
    coord_features, coord_assignments, coord_summary = aggregate_coordinate_sources(grid_pop)
    address_features, address_assignments, address_summary = aggregate_address_sources(grid_pop)
    metadata = write_outputs(
        grid_pop,
        adm_summary,
        coord_features,
        coord_assignments,
        coord_summary,
        address_features,
        address_assignments,
        address_summary,
    )
    metadata["label"] = str(config["label"])
    metadata["sgis_size"] = SGIS_SIZE
    metadata["grid_m"] = GRID_M
    return metadata


def main() -> list[dict[str, object]]:
    shapes = read_boundary_shapes()
    results = []
    for config in GRID_RUNS:
        print(f"\n=== Running {config['label']} / {config['sgis_size']} ===")
        metadata = run_one_grid(config, shapes)
        print(json.dumps(metadata, ensure_ascii=False, indent=2))
        results.append(metadata)

    summary_path = OUTPUT_DIR / "cheonan_sgis_multi_grid_metadata.json"
    summary_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nSaved summary: {summary_path}")
    return results


In [ ]:
results = main()
results


In [ ]:
# 결과 간단 검증
for config in GRID_RUNS:
    tag = config["output_tag"]
    final_path = OUTPUT_DIR / f"cheonan_sgis_{tag}_final_features_with_population.csv"
    adm_path = OUTPUT_DIR / f"cheonan_sgis_{tag}_population_by_adm_summary.csv"
    final = pd.read_csv(final_path, encoding="utf-8-sig")
    adm = pd.read_csv(adm_path, encoding="utf-8-sig")
    print(tag, final.shape, int(final["pop_2026_est_int"].sum()), int((adm["pop_2026_adm"] - adm["pop_2026_est_int_sum"]).abs().sum()))
